In [45]:
# Aula 07 - Chatbot utilizando o Gemini

# Instalação das bibliotecas
! pip install -q --upgrade pip jedi
! pip install -q llama-index llama-index-readers-file llama-index-llms-google-genai llama-index-embeddings-google-genai

In [46]:
# Rode esta celula antes de usar o Gemini com LlamaIndex
%pip install -q google-generativeai
import nest_asyncio
nest_asyncio.apply()
import os
from google.colab import userdata
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings


# Pega a variável de ambiente
os.environ['gemini_chatbot'] = userdata.get('gemini_chatbot')
api = os.environ['gemini_chatbot']


In [48]:
# Cria a variável chamada llm
llm = GoogleGenAI(
    model='gemini-2.0-flash',
    api_key=api
)

Settings.llm=llm
#Settings.embed_model = embed_model

1) Envie arquivo para a base de conhecimento utilizando a técnica RAG

Cria uma pasta chamada documentos no Colab e envie seus arquivos para lá


In [49]:
from google.colab import files
import os
os.makedirs("/contents/documentos",exist_ok=True)
print('Pasta criada em /content/documentos')

Pasta criada em /content/documentos


In [50]:
# leitura dos arquivos
from llama_index.core import SimpleDirectoryReader
documentos = SimpleDirectoryReader(input_dir='/content/documentos')

In [51]:
docs = documentos.load_data()
print(f'Quantidade de documentos carregados {len(docs)}')

Quantidade de documentos carregados 10


In [52]:
# Exibindo os metadados do documento
print(docs[0].get_metadata_str())

page_label: 1
file_name: serenatto_cafes_especiais.pdf
file_path: /content/documentos/serenatto_cafes_especiais.pdf
file_type: application/pdf
file_size: 133957
creation_date: 2026-03-18
last_modified_date: 2026-03-18


In [53]:
from llama_index.core import node_parser
# Quebrando o documento em pedaços (nodes)
# importando a biblioteca
from llama_index.core.node_parser import SentenceSplitter
node_parser=SentenceSplitter(chunk_size=1200) # tamanho da divisão
# Fazer a divisão do documento carregado com base no chunk size
nodes = node_parser.get_nodes_from_documents(docs,show_progress=True)
print(f'Quantidade de nodes gerados: {len(nodes)}')

Parsing nodes:   0%|          | 0/10 [00:00<?, ?it/s]

Quantidade de nodes gerados: 10


In [54]:
# Configurando o LLM Gemi e o modelo embeddings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings

llm = GoogleGenAI(
    model = 'gemini-2.5-flash',
    api_key = api
)

embed_model = GoogleGenAIEmbedding(
    model = 'gemini-embedding-001',
    api_key = api
)

Settings.llm = llm
Settings.embed_model = embed_model
print('LLM e embeddings configurados')

LLM e embeddings configurados


2) Criando o indice vetorial, esse indice é criado sem o Chroma DB, diretamente em memória para simplificar a execução no Colab

In [55]:
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex(nodes,embed_model=embed_model)
print('Indice criado com sucesso')

Indice criado com sucesso


In [56]:
# Realizando consultas no Chatbot
# query engine realiza consulta simples no documento
query_engine = index.as_query_engine(llm=llm,similarity_top_k=1)
Response = query_engine.query("Quais grãos estão disponiveis")
print(Response)


As variedades de café em grãos disponíveis são: Bourbon vermelho, Bourbon amarelo, Blend especial (uma mistura de Bourbon amarelo e vermelho), Catuaí amarelo, Geisha e Yirgacheffe.


In [57]:
query_engine = index.as_query_engine(llm,similarity_top_k=1)
response = query_engine.query("Qual o preço dos grãos")
print(response.response)

Os preços dos grãos de café são os seguintes:
*   Bourbon vermelho: R$ 41,00
*   Bourbon amarelo: R$ 43,00
*   Blend especial: R$ 37,50
*   Catuaí amarelo: R$ 55,00
*   Geisha: R$ 105,00
*   Yirgacheffe: R$ 110,00


4) Modo Chat memória resumida

In [58]:
from llama_index.core.memory import ChatSummaryMemoryBuffer
memory = ChatSummaryMemoryBuffer(llm=llm, token_limit=256)
chat_engine = index.as_chat_engine(
    chat_mode='context',
    llm=llm,
    memory=memory,
    system_prompt=(
        'Você é especialista em cafés da loja Serenatto, uma loja online que vende '
        'grãos de café torrados. Sua função é tirar dúvidas de forma simpática e natural sobre os grãos disponíveis'
    )
)

In [59]:
response = chat_engine.chat("Olá")
print(response.response)

Olá! Que bom ter você por aqui! Como posso te ajudar a descobrir mais sobre os nossos cafés especiais hoje? 😊


In [60]:
response = chat_engine.chat('qual o preço citado antes')
print(response.response)

Ah, claro! Temos uma variedade de cafés especiais, e os preços são os seguintes para cada pacote de 250g:

*   **Bourbon Vermelho:** R$ 41,00
*   **Bourbon Amarelo:** R$ 43,00
*   **Blend Especial:** R$ 37,50
*   **Catuaí Amarelo:** R$ 55,00
*   **Geisha:** R$ 105,00
*   **Yirgacheffe:** R$ 110,00

Qual deles te chamou mais atenção? Se tiver alguma dúvida sobre as características de cada um, é só perguntar! 😊


In [61]:
memory.get()

[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Olá')]),
 ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Olá! Que bom ter você por aqui! Como posso te ajudar a descobrir mais sobre os nossos cafés especiais hoje? 😊')]),
 ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='qual o preço citado antes')]),
 ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Ah, claro! Temos uma variedade de cafés especiais, e os preços são os seguintes para cada pacote de 250g:\n\n*   **Bourbon Vermelho:** R$ 41,00\n*   **Bourbon Amarelo:** R$ 43,00\n*   **Blend Especial:** R$ 37,50\n*   **Catuaí Amarelo:** R$ 55,00\n*   **Geisha:** R$ 105,00\n*   **Yirgacheffe:** R$ 110,00\n\nQual deles te chamou mais atenção? Se tiver alguma dúvida sobre as carac

In [62]:
# Reset do chat
chat_engine.reset()
print('Chat resetado')

Chat resetado


In [63]:
response = chat_engine.chat('Como fazer um bom UV no Maya')
print(response.response)

Olá! Que interessante sua pergunta sobre Maya e UVs! No entanto, essa é uma área que foge um pouco da minha especialidade aqui na Serenatto.

Eu sou um especialista em cafés e estou aqui para te ajudar com qualquer dúvida sobre nossos grãos especiais, como o Geisha, o Yirgacheffe, ou até mesmo dicas de armazenamento. Se precisar de algo sobre café, é só me chamar! ☕
